## tl;dr
All retained pilot/ablation answers pass. The initial specialist pilot favors LITS for this workload; the HRT-LI mutation refinement improves all three paired development runs. Neither establishes a submission-grade cross-index ranking. No 200M performance number is changed.

## Context & Methods
One million operations: 50% membership, 25% successful inserts and 25% successful deletes. Natural keys are systematically sampled across the verified 200M parent; 80/20 base/arrival split. Fresh processes, CPU 2, optimized C++17. Mutation ablation alternates order across three pairs.

### Key Assumptions
One fixed trace; process pairs, not operations, are replications. Build and whole-process RSS include adapter/harness overhead. Host counters are sampled about every five seconds and cannot establish absence of transient paging. No confidence interval is inferred from the one-pair specialist pilots.

## Data
Records under `results_q1/lits_scale_v2_20260905` and `results_q1/learned_mutation_20260905`; host logs under `results_q1/reassessment_20260905`. Full parent hashes were checked by the preparation run. Missing historical build identities remain unavailable in the provenance supplement.

In [1]:
from pathlib import Path
import json, csv, hashlib, statistics
from datetime import datetime
root = Path.cwd()
assert (root / "hrtli_cpp").is_dir()
pilot = root / "results_q1/lits_scale_v2_20260905"
mutation = root / "results_q1/learned_mutation_20260905"
prep = json.loads((pilot / "preparation.json").read_text())
assert prep["exit_code"] == 0
for scale in (1000000, 10000000):
    p = json.loads((pilot / f"corpus_{scale}.json").read_text())
    assert p["sample_keys"] == scale and p["base_keys"] + p["arrival_keys"] == scale
    assert p["parent_selected_keys"] == 200000000 and p["full_parent_order_and_hashes_verified"]
    assert p["sample_stride"] * scale == 200000000
    assert p["parent_sources_sha256"] == {
        "tmp/commoncrawl-2026-may-jun-jul-local/prepared_interleaved/commoncrawl_hosts_200m_initial.txt": "6840d31de6bfe01fd742f651298b29f0b62e6ca8ce218187a667dcbb542eeab7",
        "tmp/commoncrawl-2026-may-jun-jul-local/prepared_interleaved/commoncrawl_hosts_200m_insert.txt": "aff541a159e1f012a6034a0afbeb06eaa928f7d776c0124d6618d56bd0f15d38"}
print("PASS preparation exit, parent source digests and sample cardinalities")

PASS preparation exit, parent source digests and sample cardinalities


## Results
### Pair validity and resource checks

In [2]:
fields = ("base_keys", "arrival_keys", "operations", "workload_family", "trace_fnv64", "seed", "epsilon", "reads", "read_hits", "insert_successes", "delete_successes", "final_live_keys")
def read_record(directory, name):
    record = json.loads((directory / (name + ".json")).read_text())
    r = record["result"]
    assert record["exit_code"] == 0 and r["all_answers_checked"] and r["initial_and_final_states_checked"]
    assert r["operations"] == 1000000 and r["reads"] == 500000
    assert r["insert_attempts"] == r["insert_successes"] == 250000
    assert r["delete_attempts"] == r["delete_successes"] == 250000
    assert r["final_live_keys"] == r["base_keys"]
    swap_name = name.removeprefix("pilot_")
    assert (directory / (swap_name + "_swap_before.txt")).read_text() == (directory / (swap_name + "_swap_after.txt")).read_text()
    assert r["workload_major_faults"] == 0
    return record
def compare(a, b):
    assert all(a["result"][f] == b["result"][f] for f in fields)
    # Identical corpus hashes; binaries/source variants intentionally differ in the ablation.
    for suffix in ("/base.txt", "/base.nul", "/arrivals.nul", "/provenance.json"):
        left = [v for k,v in a["sources_sha256"].items() if k.endswith(suffix)]
        right = [v for k,v in b["sources_sha256"].items() if k.endswith(suffix)]
        assert len(left) == 1 and left == right
pairs = []
for scale in (1000000, 10000000):
    a, b = [read_record(pilot, f"pilot_{scale}_{mode}") for mode in ("lits", "hrtli")]
    compare(a,b)
    pairs.append((f"pilot_{scale}", a, b))
    print(scale, "LITS seconds", a["result"]["workload_seconds"], "HRT-LI seconds", b["result"]["workload_seconds"])
ratios = []
for trial in (1,2,3):
    a, b = [read_record(mutation, f"{trial}_{variant}") for variant in ("binary_mutation", "learned_mutation")]
    compare(a,b)
    ratio = a["result"]["workload_seconds"] / b["result"]["workload_seconds"]
    ratios.append(ratio)
    pairs.append((f"mutation_{trial}", a, b))
print("Mutation paired binary/learned ratios:", ratios, "median:", statistics.median(ratios))
print("All 10 million operation answers and 164 million initial/final memberships checked across pilot and ablation")

1000000 LITS seconds 0.571377174 HRT-LI seconds 3.261492736
10000000 LITS seconds 0.813570478 HRT-LI seconds 5.565734121
Mutation paired binary/learned ratios: [1.1004684503881776, 1.2523099128364277, 1.1573263018152846] median: 1.1573263018152846
All 10 million operation answers and 164 million initial/final memberships checked across pilot and ablation


In [3]:
resource_summary = []
for prefix in ("lits_scale_pilot_v2", "learned_mutation_ablation"):
    path = root / "results_q1/reassessment_20260905" / (prefix + ".host.csv")
    with path.open(encoding="utf-8-sig", newline="") as f:
        samples = list(csv.DictReader(f))
    times = [datetime.fromisoformat(s["Utc"].replace("Z", "+00:00")).timestamp() for s in samples]
    assert times == sorted(times) and len(times) >= 2
    result = {"session": prefix, "samples": len(samples),
              "minimum_available_MiB": min(int(s["AvailableMiB"]) for s in samples),
              "pageout_positive_samples": sum(float(s["PagesOutputPerSecond"]) > 0 for s in samples),
              "maximum_sampling_gap_seconds": max(b-a for a,b in zip(times, times[1:]))}
    resource_summary.append(result)
print(json.dumps(resource_summary, indent=2))

[
  {
    "session": "lits_scale_pilot_v2",
    "samples": 72,
    "minimum_available_MiB": 2801,
    "pageout_positive_samples": 1,
    "maximum_sampling_gap_seconds": 5.315186977386475
  },
  {
    "session": "learned_mutation_ablation",
    "samples": 19,
    "minimum_available_MiB": 2183,
    "pageout_positive_samples": 0,
    "maximum_sampling_gap_seconds": 5.3386640548706055
  }
]


## Takeaways
Share with caveats as development evidence. The negative LITS pilot is retained, not hidden. The mutation refinement is a local improvement, not proof that the specialist gap is closed. No guest swap change or timed major faults were observed in these records; sampled host counters do not prove that transient pressure was absent. Systematic samples are not independent random corpora. Prioritize current-default ART/HOT, repeated specialist comparisons, workload/split sweeps and index-only memory accounting. The separate 200M source and existing results remain unchanged.